In [ ]:
import pandas as pd
from sklearn.utils import shuffle
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler

In [ ]:
esa_21_22_corners_parsed = pd.read_pickle('data/corners_xgb_21_22_0613_2.pkl')
esa_22_23_corners_parsed = pd.read_pickle('data/corners_xgb_22_23_0613_2.pkl')
euro_corners_parsed = pd.read_pickle('data/corners_xgb_euro_0613_2.pkl')
fifa_corners_parsed = pd.read_pickle('data/corners_xgb_fifa_2022_0613_2.pkl')

esa_21_22_core = pd.read_pickle('data/38_108.pkl')
esa_22_23_core = pd.read_pickle('data/38_235.pkl')

In [ ]:
def get_test_set(df, df_core):
    dfs_test = []
    dfs_train = []
    teams = list(set(df['team'].values))
    test_matches = df_core[esa_22_23_core['match_week'] >= 18]['match_id'].values
    train_matches = df_core[esa_22_23_core['match_week'] < 18]['match_id'].values
    for team in teams:
        for match_id in test_matches:
            df_result = df[(df['team'] == team) & (df['match_id'] == match_id)]
            dfs_test.append(df_result)
        for match_id in train_matches:
            df_result = df[(df['team'] == team) & (df['match_id'] == match_id)]
            dfs_train.append(df_result)
    df_test = pd.concat(dfs_test)
    df_train = pd.concat(dfs_train)
    return df_test, df_train


esa_22_23_test, esa_22_23_train = get_test_set(esa_22_23_corners_parsed, esa_22_23_core)
print(f'Length of 22/23 test set: {len(esa_22_23_test)}')
print(f'Length of 22/23 train set: {len(esa_22_23_train)}')

In [ ]:
# baza
# esa_21_22_corners_parsed = pd.read_pickle('data/corners_xgb_21_22.pkl')
# esa_22_23_corners_parsed = pd.read_pickle('data/corners_xgb_22_23.pkl')

# baza + kolumny z liczbą podań do poszczególnych stref
# esa_21_22_corners_parsed = pd.read_pickle('data/corners_xgb_21_22_more_columns.pkl')
# esa_22_23_corners_parsed = pd.read_pickle('data/corners_xgb_22_23_more_columns.pkl')

# baza + kolumny z liczbą podań do poszczególnych stref + kolumna z liczbą strzałów na bramkę
# esa_21_22_corners_parsed = pd.read_pickle('data/corners_xgb_21_22_more_columns_on_goal.pkl')
# esa_22_23_corners_parsed = pd.read_pickle('data/corners_xgb_22_23_more_columns_on_goal.pkl')

# baza + kolumny z liczbą podań do poszczególnych stref + kolumna z liczbą strzałów na bramkę  + pass III -0.2
# esa_21_22_corners_parsed = pd.read_pickle('data/corners_xgb_21_22_0613_2.pkl')
# esa_22_23_corners_parsed = pd.read_pickle('data/corners_xgb_22_23_0613_2.pkl')


In [ ]:
test_set = esa_22_23_test
train_set = pd.concat([esa_22_23_train, esa_21_22_corners_parsed, euro_corners_parsed, fifa_corners_parsed])
print(f'Length of test set: {len(test_set)}')
print(f'Length of train set: {len(train_set)}')

In [ ]:
# test_set['was_shot'] = [1 if (shots > 0) else 0 for shots in test_set['num_of_shot'].values]
# print(f'Number of was_shots in test set: {sum(test_set["was_shot"].values)}')

# train_set['was_shot'] = [1 if (shots > 0) else 0 for shots in train_set['num_of_shot'].values]
# print(f'Number of was_shots in train set: {sum(train_set["was_shot"].values)}')

In [ ]:
test_set.columns

In [ ]:
train_set.columns

In [ ]:
# columns_to_drop = ['id', 'team', 'was_shot', 'corner_score', 'obv', 'xg']
columns_to_drop = ['id','match_id', 'team', 'corner_score', 'obv', 'xg', 'shot_on_goal']

In [ ]:
def get_sets_for_model(df_train, df_test, columns_to_drop):
    df_train = shuffle(df_train)
    df_test = shuffle(df_test)

    # scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))

    df_train_ids = df_train['id'].values  # type: ignore
    df_test_ids = df_test['id'].values  # type: ignore
    df_train_teams = df_train['team'].values  # type: ignore
    df_test_teams = df_test['team'].values  # type: ignore
    # df_train_was_shot = df_train['was_shot'].values  # type: ignore
    # df_test_was_shot = df_test['was_shot'].values  # type: ignore
    df_train_was_shot = df_train['shot_on_goal'].values  # type: ignore
    df_test_was_shot = df_test['shot_on_goal'].values  # type: ignore

    # df_train_good_ends = df_train['good_end'].values  # type: ignore
    # df_test_good_ends = df_test['good_end'].values  # type: ignore

    df_train = df_train.drop(columns=columns_to_drop) # type: ignore
    df_test = df_test.drop(columns=columns_to_drop) # type: ignore

    df_train_columns = df_train.columns.sort_values()
    df_test_columns = df_test.columns.sort_values()

    # df_train = scaler.fit_transform(df_train)
    # df_test = scaler.fit_transform(df_test)

    df_train = pd.DataFrame(df_train, columns=df_train_columns)
    df_test = pd.DataFrame(df_test, columns=df_test_columns)

    df_train['id'] = df_train_ids
    df_test['id'] = df_test_ids
    df_train['team'] = df_train_teams
    df_test['team'] = df_test_teams
    df_train['was_shot'] = df_train_was_shot
    df_test['was_shot'] = df_test_was_shot

    y_train = df_train['was_shot']
    X_train = df_train.drop(columns=['was_shot'])

    y_test = df_test['was_shot']
    X_test = df_test.drop(columns=['was_shot'])

    return X_train, X_test, y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = get_sets_for_model(train_set, test_set, columns_to_drop)

In [ ]:
from xgboost import XGBClassifier

def train_model(X_train, X_test, y_train):
    model = XGBClassifier(objective='binary:logistic')

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot'])
    model.fit(input_X_train, y_train)

    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot'])
    XGB_predictions = model.predict(input_X_test)

    return model, XGB_predictions, input_X_test

model, pred, input_test = train_model(X_train, X_test, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def evaluate_model(y_test, pred):
    print('Confusion matrix')
    print(confusion_matrix(y_test, pred))
    print('Classification report')
    print(classification_report(y_test, pred))
    print('Accuracy= ', accuracy_score(y_test, pred))

evaluate_model(y_test, pred)

In [ ]:
from datetime import datetime

# datetime object containing current date and time
now = datetime.now()
 
print("now =", now)

# dd/mm/YY H:M:S
dt_string = now.strftime("%d%m%Y_%H%M%S")
print("date and time =", dt_string)

In [ ]:
def feature_importances_raport(input_X_test, model, set_title):
    result = pd.DataFrame(columns=['name', 'value'])
    result['name'] = input_X_test.columns
    result['value'] = model.feature_importances_

    result = result.sort_values(by=['value'], ascending=False)

    result.to_csv(f'features_{set_title}.csv')
    print(result)
    return result

feature_importances = feature_importances_raport(input_test, model, f'corners_experiment_{dt_string}')

In [ ]:
def generate_scores(model, X_test, X_train, y_train, set_title):
    model = XGBClassifier(objective='binary:logistic')

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot'])
    # input_X_train = X_train.drop(columns=['id', 'team', 'shot_on_goal', 'num_of_shot'])
    model.fit(input_X_train, y_train)
    
    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot'])
    # input_X_test = X_test.drop(columns=['id', 'team', 'shot_on_goal', 'num_of_shot'])
    model = model.predict_proba(input_X_test)

    X_test['score_positive'] = [m[1] for m in model]
    X_test['score_negative'] = [m[0] for m in model]

    X_test.to_csv(f'{set_title}_evaluated.csv')

    return X_test

df_with_scores = generate_scores(model, X_test, X_train, y_train, f'corners_experiment_{dt_string}')

In [ ]:
def show_correlation(df_with_scores, test_set):
    df_with_scores = df_with_scores.merge(test_set[['id','xg']], on='id')
    df_with_scores = df_with_scores.merge(test_set[['id','obv']], on='id')
    df_with_scores = df_with_scores.merge(test_set[['id','corner_score']], on='id')
    corr_with_xg = df_with_scores['score_positive'].corr(df_with_scores['xg'])
    corr_with_obv = df_with_scores['score_positive'].corr(df_with_scores['obv'])
    corr_with_corner_score = df_with_scores['score_positive'].corr(df_with_scores['corner_score'])
    print(f'Correlation with xg: {corr_with_xg:.3f}')
    print(f'Correlation with obv: {corr_with_obv:.3f}')
    print(f'Correlation with corner_score: {corr_with_corner_score:.3f}')
    return df_with_scores

In [ ]:
df_with_scores = show_correlation(df_with_scores, test_set)

**Corners: default columns**\
Correlation with xg: 0.462\
Correlation with obv: 0.150\
Correlation with corner_score: 0.341


**Corners: columns with num to every zone passes**\
Correlation with xg: 0.463\
Correlation with obv: 0.192\
Correlation with corner_score: 0.345


**Corners: columns with num to every zone passes and label is shot on goal**\
Correlation with xg: 0.469\
Correlation with obv: 0.156\
Correlation with corner_score: 0.268

In [ ]:
def sum_metric_for_each_team(df_with_scores, metric):
    df_with_scores = df_with_scores.groupby(['team']).agg({metric: 'sum'}).reset_index()
    df_with_scores = df_with_scores.sort_values(by=[metric], ascending=False)
    return df_with_scores

In [ ]:
def avg_metric_for_each_team(df_with_scores, metric):
    df_with_scores = df_with_scores.groupby(['team']).agg({metric: 'mean'}).reset_index()
    df_with_scores = df_with_scores.sort_values(by=[metric], ascending=False)
    return df_with_scores

In [ ]:
avg_metric_for_each_team(df_with_scores, 'score_positive')

In [ ]:
avg_metric_for_each_team(df_with_scores, 'corner_score')

In [ ]:
avg_metric_for_each_team(df_with_scores, 'xg')

In [ ]:
def get_avg_metrics_for_each_team(df_with_scores, test_set):
    df_with_scores = df_with_scores.groupby(['team']).agg({'score_positive': 'mean', 'obv': 'mean', 'xg': 'mean', 'corner_score' : 'mean'}).reset_index()
    df_with_scores = df_with_scores.sort_values(by=['score_positive'], ascending=False)
    return df_with_scores

def get_sum_metrics_for_each_team(df_with_scores, test_set):
    df_with_scores = df_with_scores.groupby(['team']).agg({'score_positive': 'sum', 'obv': 'sum', 'xg': 'sum', 'corner_score' : 'sum'}).reset_index()
    df_with_scores = df_with_scores.sort_values(by=['score_positive'], ascending=False)
    return df_with_scores

def get_med_metrics_for_each_team(df_with_scores, test_set):
    df_with_scores = df_with_scores.groupby(['team']).agg({'score_positive': 'median', 'obv': 'median', 'xg': 'median', 'corner_score' : 'median'}).reset_index()
    df_with_scores = df_with_scores.sort_values(by=['score_positive'], ascending=False)
    return df_with_scores





In [ ]:
res_avg = get_avg_metrics_for_each_team(df_with_scores, test_set)
res_sum = get_sum_metrics_for_each_team(df_with_scores, test_set)
res_med = get_med_metrics_for_each_team(df_with_scores, test_set)

In [ ]:
res.to_csv('./results/avg_metrics_label_is_shot_on_goal.csv')

In [ ]:
res_sum